# XGBoost from feature extraction

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import xgboost as xgb

from feature_extraction import extract_features_from_df_video


In [2]:

df_video = pd.read_parquet("../data/filled_gait_data_encoded.parquet")
df_video.head()


,id,patient_name,frame,movement_type,side,timestamp_ms,landmark_id,x_norm,y_norm,z_norm,...,gait_pattern,add_pattern_info,title,fps,width,height,gait_markers,file_path,video_id,dataset_encoded
0,1107969,PA027,51,Fast Movement,Right,1700.0,5,0.659191,0.317202,-0.080875,...,normal,normal,normal,33.0,960,540,None,C:\User_V\2_Github\GAITy-Capstone-Modeling\dat...,semantic_segmentation_PA027_FGS_WJ_1_DensePose...,0
1,1107975,PA027,51,Fast Movement,Right,1700.0,11,0.680979,0.388419,-0.135898,...,normal,normal,normal,33.0,960,540,None,C:\User_V\2_Github\GAITy-Capstone-Modeling\dat...,semantic_segmentation_PA027_FGS_WJ_1_DensePose...,0
2,1107976,PA027,51,Fast Movement,Right,1700.0,12,0.661917,0.387589,0.021804,...,normal,normal,normal,33.0,960,540,None,C:\User_V\2_Github\GAITy-Capstone-Modeling\dat...,semantic_segmentation_PA027_FGS_WJ_1_DensePose...,0
3,1107987,PA027,51,Fast Movement,Right,1700.0,23,0.662738,0.550898,-0.054501,...,normal,normal,normal,33.0,960,540,None,C:\User_V\2_Github\GAITy-Capstone-Modeling\dat...,semantic_segmentation_PA027_FGS_WJ_1_DensePose...,0
4,1107988,PA027,51,Fast Movement,Right,1700.0,24,0.667992,0.545544,0.054363,...,normal,normal,normal,33.0,960,540,None,C:\User_V\2_Github\GAITy-Capstone-Modeling\dat...,semantic_segmentation_PA027_FGS_WJ_1_DensePose...,0


In [3]:
import sys
sys.path.append("..")  # falls nötig, damit feature_extraction gefunden wird

import pandas as pd
from feature_extraction import build_df_video, extract_features_from_df_video


In [4]:
df_raw = pd.read_parquet("../data/filled_gait_data_encoded.parquet")
df_raw.head()


,id,patient_name,frame,movement_type,side,timestamp_ms,landmark_id,x_norm,y_norm,z_norm,...,gait_pattern,add_pattern_info,title,fps,width,height,gait_markers,file_path,video_id,dataset_encoded
0,1107969,PA027,51,Fast Movement,Right,1700.0,5,0.659191,0.317202,-0.080875,...,normal,normal,normal,33.0,960,540,None,C:\User_V\2_Github\GAITy-Capstone-Modeling\dat...,semantic_segmentation_PA027_FGS_WJ_1_DensePose...,0
1,1107975,PA027,51,Fast Movement,Right,1700.0,11,0.680979,0.388419,-0.135898,...,normal,normal,normal,33.0,960,540,None,C:\User_V\2_Github\GAITy-Capstone-Modeling\dat...,semantic_segmentation_PA027_FGS_WJ_1_DensePose...,0
2,1107976,PA027,51,Fast Movement,Right,1700.0,12,0.661917,0.387589,0.021804,...,normal,normal,normal,33.0,960,540,None,C:\User_V\2_Github\GAITy-Capstone-Modeling\dat...,semantic_segmentation_PA027_FGS_WJ_1_DensePose...,0
3,1107987,PA027,51,Fast Movement,Right,1700.0,23,0.662738,0.550898,-0.054501,...,normal,normal,normal,33.0,960,540,None,C:\User_V\2_Github\GAITy-Capstone-Modeling\dat...,semantic_segmentation_PA027_FGS_WJ_1_DensePose...,0
4,1107988,PA027,51,Fast Movement,Right,1700.0,24,0.667992,0.545544,0.054363,...,normal,normal,normal,33.0,960,540,None,C:\User_V\2_Github\GAITy-Capstone-Modeling\dat...,semantic_segmentation_PA027_FGS_WJ_1_DensePose...,0


In [5]:
# mappen von file_path zu source_file
if "source_file" not in df_raw.columns and "file_path" in df_raw.columns:
    df_raw = df_raw.rename(columns={"file_path": "source_file"})


In [6]:
import pandas as pd

df_raw = pd.read_parquet("../data/filled_gait_data_encoded.parquet")

# falls noch nicht gemacht: file_path -> source_file umbenennen
if "source_file" not in df_raw.columns and "file_path" in df_raw.columns:
    df_raw = df_raw.rename(columns={"file_path": "source_file"})

# Frames integer machen
df_raw["frame"] = df_raw["frame"].astype(int)

# FRAMES PRO VIDEO AUF 0 BEGINNEN LASSEN
# (wichtig, weil add_pose_column davon ausgeht, dass frame bei 0 startet)
df_raw["frame"] = df_raw["frame"] - df_raw.groupby("video_id")["frame"].transform("min")


In [7]:
df_raw.groupby("video_id")["frame"].agg(["min", "max"]).head()


,min,max
video_id,,
cljar878f00c03n6ly2v2ay88_right side_nan_Abnormal Gait_cerebral palsy_landmarks,0,192
cljar9bqo00c43n6l2u5zmlru_left side_nan_Abnormal Gait_cerebral palsy_landmarks,0,193
cljar9t8o00c83n6ltculhoct_right side_nan_Abnormal Gait_cerebral palsy_landmarks,0,639
cljarar9t00cc3n6lqhi9udoc_left side_nan_Abnormal Gait_cerebral palsy_landmarks,0,762
cljarbn1y00cg3n6l1u4i0d5l_front_nan_Abnormal Gait_cerebral palsy_landmarks,0,150


In [9]:
import pandas as pd

# neu laden
df_raw = pd.read_parquet("../data/filled_gait_data_encoded.parquet")

# file_path -> source_file, falls noch nicht vorhanden
if "source_file" not in df_raw.columns and "file_path" in df_raw.columns:
    df_raw = df_raw.rename(columns={"file_path": "source_file"})

# sicherstellen, dass frame integer ist
df_raw["frame"] = df_raw["frame"].astype(int)

# 🔴 WICHTIG: Frames pro source_file auf 0-basierend normalisieren
df_raw["frame"] = df_raw["frame"] - df_raw.groupby("source_file")["frame"].transform("min")

# kurz prüfen, ob das geklappt hat
df_raw.groupby("source_file")["frame"].agg(["min", "max"]).head()


,min,max
source_file,,
C:\User_V\2_Github\GAITy-Capstone-Modeling\data\csv\Health_Gait_0_397\semantic_segmentation_PA000_FGS_WJ_1_DensePose_landmarks.csv,0,165
C:\User_V\2_Github\GAITy-Capstone-Modeling\data\csv\Health_Gait_0_397\semantic_segmentation_PA000_FGS_WJ_2_DensePose_landmarks.csv,0,163
C:\User_V\2_Github\GAITy-Capstone-Modeling\data\csv\Health_Gait_0_397\semantic_segmentation_PA000_FGS_WoJ_1_DensePose_landmarks.csv,0,141
C:\User_V\2_Github\GAITy-Capstone-Modeling\data\csv\Health_Gait_0_397\semantic_segmentation_PA000_FGS_WoJ_2_DensePose_landmarks.csv,0,166
C:\User_V\2_Github\GAITy-Capstone-Modeling\data\csv\Health_Gait_0_397\semantic_segmentation_PA000_UGS_WJ_1_DensePose_landmarks.csv,0,167


In [20]:
import numpy as np
import pandas as pd

N_JOINTS = 33  # MediaPipe Pose

def build_df_video_from_raw(df_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Baut aus dem long-format df_raw (eine Zeile pro Frame+Landmark) ein df_video:
    - eine Zeile pro Video/Clip (pro source_file)
    - Spalte 'pose' mit Shape (T, 33, 3)
    - plus Meta-Infos wie fps, gait_pattern, movement_type, side, source_file, dataset_encoded, ...
    """

    df = df_raw.copy()
    df["frame"] = df["frame"].astype(int)
    df["landmark_id"] = df["landmark_id"].astype(int)

    meta_cols = [
        "source_file",
        "fps",
        "gait_pattern",
        "movement_type",
        "side",
        "patient_name",
        "video_id",
        "dataset",
        "dataset_encoded",   # 🔴 WICHTIG hinzugefügt
        "add_pattern_info",
        "title",
        "gait_markers",
    ]
    meta_cols = [c for c in meta_cols if c in df.columns]

    rows = []

    for _, group in df.groupby("source_file"):
        group = group.sort_values(["frame", "landmark_id"])

        # Frames robust auf 0..T-1 mappen
        unique_frames = np.sort(group["frame"].unique())
        frame_to_idx = {f: i for i, f in enumerate(unique_frames)}
        T = len(unique_frames)

        pose = np.zeros((T, N_JOINTS, 3), dtype=float)

        for _, r in group.iterrows():
            f_idx = frame_to_idx[int(r["frame"])]
            j = int(r["landmark_id"])
            if 0 <= j < N_JOINTS:
                pose[f_idx, j, 0] = r["x_norm"]
                pose[f_idx, j, 1] = r["y_norm"]
                pose[f_idx, j, 2] = r["z_norm"]

        base = group.iloc[0][meta_cols].copy()
        base["pose"] = pose
        rows.append(base)

    return pd.DataFrame(rows)


In [21]:
from feature_extraction import extract_features_from_df_video

# a) Parquet laden
df_raw = pd.read_parquet("../data/filled_gait_data_encoded.parquet")

# file_path -> source_file, falls noch nötig
if "source_file" not in df_raw.columns and "file_path" in df_raw.columns:
    df_raw = df_raw.rename(columns={"file_path": "source_file"})

# b) unser robustes df_video bauen
df_video = build_df_video_from_raw(df_raw)
print(df_video.shape)
df_video.head()


(3279, 13)


,source_file,fps,gait_pattern,movement_type,side,patient_name,video_id,dataset,dataset_encoded,add_pattern_info,title,gait_markers,pose
374434,C:\User_V\2_Github\GAITy-Capstone-Modeling\dat...,33.0,normal,Fast Movement,Right,PA000,semantic_segmentation_PA000_FGS_WJ_1_DensePose...,normal,0,normal,normal,None,"[[[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.9155808..."
388630,C:\User_V\2_Github\GAITy-Capstone-Modeling\dat...,33.0,normal,Fast Movement,Left,PA000,semantic_segmentation_PA000_FGS_WJ_2_DensePose...,normal,0,normal,normal,None,"[[[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0256412..."
393194,C:\User_V\2_Github\GAITy-Capstone-Modeling\dat...,33.0,normal,Fast Movement,Right,PA000,semantic_segmentation_PA000_FGS_WoJ_1_DensePos...,normal,0,normal,normal,None,"[[[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.9018083..."
390912,C:\User_V\2_Github\GAITy-Capstone-Modeling\dat...,33.0,normal,Fast Movement,Left,PA000,semantic_segmentation_PA000_FGS_WoJ_2_DensePos...,normal,0,normal,normal,None,"[[[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0200935..."
460713,C:\User_V\2_Github\GAITy-Capstone-Modeling\dat...,33.0,normal,Regular Movement,Right,PA000,semantic_segmentation_PA000_UGS_WJ_1_DensePose...,normal,0,normal,normal,None,"[[[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.9318467..."


In [22]:
df_video.columns
# u.a.: 'pose', 'fps', 'gait_pattern', 'movement_type', 'side', 'source_file', ...


Index(['source_file', 'fps', 'gait_pattern', 'movement_type', 'side',
       'patient_name', 'video_id', 'dataset', 'dataset_encoded',
       'add_pattern_info', 'title', 'gait_markers', 'pose'],
      dtype='object')

In [23]:
df_features = extract_features_from_df_video(df_video)
df_features.shape, df_features.head()


((3279, 88),
    step_height_L  step_height_R  step_length_L  step_length_R   
 0       1.333319       1.181641       1.164251       0.984652  \
 1       2.263144       2.727832       1.365075       1.441705   
 2       0.998986       0.937598       1.023654       1.003390   
 3       1.873531       2.517548       1.436746       1.432255   
 4       0.503940       0.477920       0.992774       0.861207   
 
    pelvis_drop_mean  pelvis_drop_std  trunk_lean_mean  trunk_lean_std   
 0          0.020217         0.021468        -0.030305        0.136898  \
 1          0.015201         0.078882        -0.054901        0.105949   
 2          0.011532         0.019520         0.008385        0.144712   
 3          0.001384         0.064399        -0.039065        0.130419   
 4          0.018124         0.016102        -0.041142        0.137960   
 
    heel_range_L  heel_range_R  ...  step_time_asym  cadence_asym   
 0      1.435486      1.271801  ...       -0.046607      0.046607  \
 1   

In [28]:
import pandas as pd
from feature_extraction import extract_features_from_df_video

# a) Parquet laden
df_raw = pd.read_parquet("../data/filled_gait_data_encoded.parquet")

if "source_file" not in df_raw.columns and "file_path" in df_raw.columns:
    df_raw = df_raw.rename(columns={"file_path": "source_file"})

# b) df_video bauen
df_video = build_df_video_from_raw(df_raw)
print(df_video.shape)
print(df_video.columns)

# c) Features extrahieren
df_features = extract_features_from_df_video(df_video)
print(df_features.shape)
print(df_features.columns)

# Labels checken
print(df_features[["label_fine", "label_class", "label_id"]].head())
print(df_features["label_id"].value_counts(dropna=False))


(3279, 13)
Index(['source_file', 'fps', 'gait_pattern', 'movement_type', 'side',
       'patient_name', 'video_id', 'dataset', 'dataset_encoded',
       'add_pattern_info', 'title', 'gait_markers', 'pose'],
      dtype='object')
(3279, 88)
Index(['step_height_L', 'step_height_R', 'step_length_L', 'step_length_R',
       'pelvis_drop_mean', 'pelvis_drop_std', 'trunk_lean_mean',
       'trunk_lean_std', 'heel_range_L', 'heel_range_R',
       'step_height_symmetry', 'step_length_symmetry',
       'knee_L_moving_time_sec', 'knee_L_still_time_sec',
       'knee_L_moving_fraction', 'knee_L_still_fraction', 'knee_L_mean_speed',
       'knee_L_max_speed', 'knee_L_total_time_sec', 'knee_R_moving_time_sec',
       'knee_R_still_time_sec', 'knee_R_moving_fraction',
       'knee_R_still_fraction', 'knee_R_mean_speed', 'knee_R_max_speed',
       'knee_R_total_time_sec', 'knee_L_rom_y', 'knee_R_rom_y', 'hip_L_rom_y',
       'hip_R_rom_y', 'shoulder_L_rom_x', 'shoulder_R_rom_x', 'ankle_L_rom_y',
    

In [29]:
non_feature_cols = [
    "label_fine",
    "label_class",
    "label_id",
    "movement_type",
    "side",
    "source_file",
]

df_ml = df_features.dropna(subset=["label_id"]).copy()
df_ml["label_id"] = df_ml["label_id"].astype(int)

feature_cols = [
    c for c in df_ml.columns
    if c not in non_feature_cols and pd.api.types.is_numeric_dtype(df_ml[c])
]

X = df_ml[feature_cols].fillna(df_ml[feature_cols].median(numeric_only=True))
y = df_ml["label_id"]

print(X.shape)
print(y.shape)
print(y.value_counts())


(0, 82)
(0,)
Series([], Name: count, dtype: int64)


In [30]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import numpy as np

# df_features kommt aus extract_features_from_df_video(df_video)

# Nur Zeilen mit gültigem fine label behalten
df_ml = df_features.dropna(subset=["label_fine"]).copy()

# String-Labels encoden
le = LabelEncoder()
df_ml["label_encoded"] = le.fit_transform(df_ml["label_fine"].astype(str))

non_feature_cols = [
    "label_fine",
    "label_class",
    "label_id",         # benutzen wir jetzt NICHT mehr
    "label_encoded",    # kommt in y, nicht in X
    "movement_type",
    "side",
    "source_file",
]

feature_cols = [
    c for c in df_ml.columns
    if c not in non_feature_cols and pd.api.types.is_numeric_dtype(df_ml[c])
]

X = df_ml[feature_cols].fillna(df_ml[feature_cols].median(numeric_only=True))
y = df_ml["label_encoded"]

print(X.shape)
print(y.shape)
print(y.value_counts())


(3279, 82)
(3279,)
label_encoded
10    3067
9       75
5       24
3       24
11      18
8       14
7       14
6       12
1       10
2        8
0        8
4        5
Name: count, dtype: int64


In [31]:
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

xgb_clf = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=len(np.unique(y)),
    eval_metric="mlogloss",
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)

xgb_clf.fit(X_train, y_train)

y_pred = xgb_clf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=le.classes_))
print(confusion_matrix(y_test, y_pred))


                          precision    recall  f1-score   support

       Assymetrical Gait       1.00      1.00      1.00         2
        Chronic CVA Gait       0.33      0.50      0.40         2
   Chronic Gait Dystonia       1.00      1.00      1.00         1
Chronic Hemiparetic Gait       0.80      0.80      0.80         5
Diabetic Neuropathy Gait       0.00      0.00      0.00         1
           Gait Dystonia       1.00      1.00      1.00         5
 Multiple Sclerosis Gait       0.00      0.00      0.00         2
Parkinson's Disease Gait       0.00      0.00      0.00         3
Transverse Myelitis Gait       1.00      0.33      0.50         3
                abnormal       0.62      1.00      0.77        15
                  normal       1.00      1.00      1.00       613
              prosthetic       0.00      0.00      0.00         4

                accuracy                           0.98       656
               macro avg       0.56      0.55      0.54       656
        

c:\Users\lejaz\spice_bootcamp\GAITy-Capstone-Modeling\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\lejaz\spice_bootcamp\GAITy-Capstone-Modeling\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\lejaz\spice_bootcamp\GAITy-Capstone-Modeling\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_

---

# first binary (abnormal or normal gait) baseline

In [43]:
import numpy as np
import pandas as pd

# nur Zeilen mit label_fine behalten
df_bin = df_features.dropna(subset=["label_fine"]).copy()

# optional: Prothesenfälle erstmal rausnehmen (sehr wenige & sehr anders)
df_bin = df_bin[df_bin["label_fine"].str.lower() != "prosthetic"]

# Binary-Label:
# 0 = normal, 1 = abnormal
df_bin["binary_label"] = np.where(
    df_bin["label_fine"].str.lower() == "normal",
    0,
    1
)

df_bin[["label_fine", "binary_label"]].head()
print(df_bin["binary_label"].value_counts())


binary_label
0    3067
1     194
Name: count, dtype: int64


In [45]:
# du hast ja schon:
# df_video = ...
# df_features = extract_features_from_df_video(df_video)

# Map von source_file -> patient_name bauen
df_pat = df_video[["source_file", "patient_name"]].drop_duplicates("source_file")

# in df_features hineinjoinen
df_features = df_features.merge(df_pat, on="source_file", how="left")

df_features[["source_file", "patient_name", "label_fine"]].head()


,source_file,patient_name,label_fine
0,C:\User_V\2_Github\GAITy-Capstone-Modeling\dat...,PA000,normal
1,C:\User_V\2_Github\GAITy-Capstone-Modeling\dat...,PA000,normal
2,C:\User_V\2_Github\GAITy-Capstone-Modeling\dat...,PA000,normal
3,C:\User_V\2_Github\GAITy-Capstone-Modeling\dat...,PA000,normal
4,C:\User_V\2_Github\GAITy-Capstone-Modeling\dat...,PA000,normal


In [46]:
import numpy as np
import pandas as pd

# nur Clips mit gültigem fine label
df_bin = df_features.dropna(subset=["label_fine"]).copy()

# optional: Prothesen raus
df_bin = df_bin[df_bin["label_fine"].str.lower() != "prosthetic"]

# Binary-Label: 0 = normal, 1 = abnormal
df_bin["binary_label"] = np.where(
    df_bin["label_fine"].str.lower() == "normal",
    0,
    1
)

# jetzt sollte patient_name existieren
print("Spalten:", df_bin.columns.tolist())
print("Anzahl Samples:", df_bin.shape[0])
print("Binary-Label-Verteilung:\n", df_bin["binary_label"].value_counts())
print("Anzahl Patienten:", df_bin["patient_name"].nunique())


Spalten: ['step_height_L', 'step_height_R', 'step_length_L', 'step_length_R', 'pelvis_drop_mean', 'pelvis_drop_std', 'trunk_lean_mean', 'trunk_lean_std', 'heel_range_L', 'heel_range_R', 'step_height_symmetry', 'step_length_symmetry', 'knee_L_moving_time_sec', 'knee_L_still_time_sec', 'knee_L_moving_fraction', 'knee_L_still_fraction', 'knee_L_mean_speed', 'knee_L_max_speed', 'knee_L_total_time_sec', 'knee_R_moving_time_sec', 'knee_R_still_time_sec', 'knee_R_moving_fraction', 'knee_R_still_fraction', 'knee_R_mean_speed', 'knee_R_max_speed', 'knee_R_total_time_sec', 'knee_L_rom_y', 'knee_R_rom_y', 'hip_L_rom_y', 'hip_R_rom_y', 'shoulder_L_rom_x', 'shoulder_R_rom_x', 'ankle_L_rom_y', 'ankle_R_rom_y', 'knee_rom_asym', 'hip_rom_asym', 'shoulder_rom_asym', 'ankle_rom_asym', 'ankle_L_moving_fraction', 'ankle_L_still_fraction', 'ankle_R_moving_fraction', 'ankle_R_still_fraction', 'stance_ratio_L', 'stance_ratio_R', 'stance_ratio_asym', 'knee_angle_L_mean', 'knee_angle_L_std', 'knee_angle_L_rom'

In [47]:
non_feature_cols = [
    "label_fine",
    "label_class",
    "label_id",
    "binary_label",
    "movement_type",
    "side",
    "source_file",
    "patient_name",
]

feature_cols = [
    c for c in df_bin.columns
    if c not in non_feature_cols and pd.api.types.is_numeric_dtype(df_bin[c])
]

X = df_bin[feature_cols].fillna(df_bin[feature_cols].median(numeric_only=True))
y = df_bin["binary_label"].astype(int)
groups = df_bin["patient_name"]

print("X, y shape:", X.shape, y.shape)


X, y shape: (3261, 82) (3261,)


In [48]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_idx].reset_index(drop=True)
X_test  = X.iloc[test_idx].reset_index(drop=True)
y_train = y.iloc[train_idx].reset_index(drop=True)
y_test  = y.iloc[test_idx].reset_index(drop=True)

train_pats = set(groups.iloc[train_idx])
test_pats  = set(groups.iloc[test_idx])
print("gemeinsame Patienten in Train & Test:", len(train_pats & test_pats))


gemeinsame Patienten in Train & Test: 0


In [49]:
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import numpy as np

ratio = (y_train == 0).sum() / (y_train == 1).sum()

xgb_bin = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=ratio,
    random_state=42,
    n_jobs=-1,
)

xgb_bin.fit(X_train, y_train)

y_pred = xgb_bin.predict(X_test)
y_proba = xgb_bin.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=["normal", "abnormal"]))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))


              precision    recall  f1-score   support

      normal       1.00      0.99      1.00       600
    abnormal       0.93      0.97      0.95        40

    accuracy                           0.99       640
   macro avg       0.96      0.98      0.97       640
weighted avg       0.99      0.99      0.99       640

Confusion matrix:
 [[597   3]
 [  1  39]]
ROC-AUC: 0.9980416666666666


Train/Test leakage check